# 02 base model 到底怎么加载：模型从哪来

今天只解决一个问题：

> `Qwen/Qwen2.5-0.5B-Instruct` 这串名字，是怎么变成电脑里的一个模型对象的？

核心入口是 `AutoModelForCausalLM.from_pretrained(...)`。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("本 micro notebook 默认只读代码；真实模型推理开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=80):
    lines = read_text(rel).splitlines()
    print(f"--- {rel}:{start}-{min(end, len(lines))} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = [i for i, line in enumerate(lines, start=1) if keyword in line]
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        show_file(rel, max(1, hit-context), min(len(lines), hit+context))
        print()

## 1. 模型名在哪里定义？

In [ ]:
find_lines("scripts/infer.py", "DEFAULT_MODEL", context=5)
find_lines("scripts/train_sft_lora.py", "--model_name", context=5)

推理脚本和训练脚本默认都用同一个 base model：`Qwen/Qwen2.5-0.5B-Instruct`。

## 2. Hugging Face 缓存在哪里？

项目运行前建议设置 `HF_HOME=.hf_cache`，这样模型会缓存到项目 `.hf_cache/`。

In [ ]:
cache = PROJECT_ROOT / ".hf_cache"
print(".hf_cache exists:", cache.exists())
if cache.exists():
    for rel in [".hf_cache/hub", ".hf_cache/datasets"]:
        p = PROJECT_ROOT / rel
        print(rel, "exists:", p.exists())

## 3. 推理时怎么加载模型？

In [ ]:
find_lines("scripts/infer.py", "AutoModelForCausalLM.from_pretrained", context=14)

关键参数解释：`args.model_name` 是模型名；`dtype` 影响显存；`device_map="auto"` 有 CUDA 时自动放到 GPU；`local_files_only` 控制是否只读本地缓存。

## 4. 训练时也要加载 base model

In [ ]:
find_lines("scripts/train_sft_lora.py", "AutoModelForCausalLM.from_pretrained", context=14)

区别是：推理加载 base model 后直接 generate；SFT 训练加载 base model 后，还要挂 LoRA adapter；DPO 加载 base + SFT adapter 后，再用偏好数据训练。

## 5. 本节面试三句话

1. Qwen base model 通过 `AutoModelForCausalLM.from_pretrained` 从 Hugging Face 缓存或网络加载。
2. `.hf_cache` 让项目可以把模型缓存固定在仓库目录下。
3. 推理、SFT、DPO 都先以 base Qwen 为底座，后续 adapter 都是挂在它上面。